<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook07_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytorch-fid==0.3.0 open-clip-torch==2.24.0 pandas -q
print("Install complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
Install complete.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MyDissertationCN6000')
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
assert torch.cuda.is_available(), "Need a GPU runtime (CLIP and Inception both run on GPU)."
print(f"GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB


In [3]:
from pytorch_fid.fid_score import calculate_fid_given_paths

def run_fid(dir_a, dir_b):
    return calculate_fid_given_paths(
        [str(dir_a), str(dir_b)],
        batch_size=50, device="cuda", dims=2048,
    )

Calculating FID 3 ways:
- baseline unrelated, erased unrelated
- baseline target, erased target
- recovered, baseline target

In [4]:
print("=== FID — Fidelity (baseline unrelated vs erased unrelated, N=250) ===")
fid_fidelity = run_fid("outputs/baseline/unrelated", "outputs/erased/unrelated")
print(f"  FID: {fid_fidelity:.4f}")

print("\n=== FID — Efficacy (baseline target vs erased target, N=50) ===")
fid_efficacy = run_fid("outputs/baseline/target", "outputs/erased/target")
print(f"  FID: {fid_efficacy:.4f}")

print("\n=== FID — Resilience (recovered vs baseline target, N=50) ===")
fid_resilience = run_fid("outputs/recovered", "outputs/baseline/target")
print(f"  FID: {fid_resilience:.4f}")

print(f"\nFID summary:")
print(f"  Fidelity:    {fid_fidelity:.4f}  (lower = unrelated content preserved)")
print(f"  Efficacy:    {fid_efficacy:.4f}  (higher = bigger change from baseline = good erasure)")
print(f"  Resilience:  {fid_resilience:.4f}  (higher = recovery failed to match baseline = robust)")

=== FID — Fidelity (baseline unrelated vs erased unrelated, N=250) ===
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 635MB/s]
100%|██████████| 5/5 [02:19<00:00, 27.83s/it]


  FID: 74.0800

=== FID — Efficacy (baseline target vs erased target, N=50) ===


100%|██████████| 1/1 [00:45<00:00, 45.97s/it]


  FID: 300.7520

=== FID — Resilience (recovered vs baseline target, N=50) ===


100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


  FID: 331.4457

FID summary:
  Fidelity:    74.0800  (lower = unrelated content preserved)
  Efficacy:    300.7520  (higher = bigger change from baseline = good erasure)
  Resilience:  331.4457  (higher = recovery failed to match baseline = robust)


Loading CLIP model

In [5]:
import open_clip
from PIL import Image
from pathlib import Path
import json

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to("cuda").eval()
print("CLIP ViT-B-32 loaded.")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


CLIP ViT-B-32 loaded.


CLIP-T (text-to-image similarity)

In [6]:
def clip_t_score(image_dir, log_entries):
    scores = []
    for entry in log_entries:
        img_path = image_dir / entry["filename"]
        image = preprocess(Image.open(img_path)).unsqueeze(0).to("cuda")
        text = clip_tokenizer([entry["prompt"]]).to("cuda")
        with torch.no_grad():
            img_f = clip_model.encode_image(image)
            txt_f = clip_model.encode_text(text)
            img_f = img_f / img_f.norm(dim=-1, keepdim=True)
            txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
            scores.append((img_f @ txt_f.T).item())
    return sum(scores) / len(scores)

baseline_log = json.loads(Path("logs/seeds_baseline.json").read_text())
erased_log = json.loads(Path("logs/seeds_erased.json").read_text())

print("=== CLIP-T on unrelated prompts ===")
clip_t_baseline_unrel = clip_t_score(Path("outputs/baseline/unrelated"), baseline_log["unrelated"])
clip_t_erased_unrel   = clip_t_score(Path("outputs/erased/unrelated"),   erased_log["unrelated"])
print(f"  Baseline:  {clip_t_baseline_unrel:.4f}")
print(f"  Erased:    {clip_t_erased_unrel:.4f}")
print(f"  Delta:     {clip_t_erased_unrel - clip_t_baseline_unrel:+.4f}")
print(f"  (Delta close to 0 = fidelity preserved; large negative = erasure damaged unrelated capability)")

=== CLIP-T on unrelated prompts ===
  Baseline:  0.3134
  Erased:    0.3096
  Delta:     -0.0038
  (Delta close to 0 = fidelity preserved; large negative = erasure damaged unrelated capability)


CLIP-I (image to image similarity)

In [7]:
def clip_i_score(dir_a, dir_b):
    """Mean CLIP cosine similarity between paired images (matched by filename)."""
    scores = []
    for path_a in sorted(dir_a.glob("*.png")):
        path_b = dir_b / path_a.name
        if not path_b.exists():
            continue
        img_a = preprocess(Image.open(path_a)).unsqueeze(0).to("cuda")
        img_b = preprocess(Image.open(path_b)).unsqueeze(0).to("cuda")
        with torch.no_grad():
            f_a = clip_model.encode_image(img_a)
            f_b = clip_model.encode_image(img_b)
            f_a = f_a / f_a.norm(dim=-1, keepdim=True)
            f_b = f_b / f_b.norm(dim=-1, keepdim=True)
            scores.append((f_a @ f_b.T).item())
    return sum(scores) / len(scores) if scores else 0.0

print("=== CLIP-I — Efficacy (erased target vs baseline target) ===")
clip_i_efficacy = clip_i_score(Path("outputs/erased/target"), Path("outputs/baseline/target"))
print(f"  CLIP-I efficacy: {clip_i_efficacy:.4f}")
print(f"  (Lower = more change from baseline = good erasure)")

print("\n=== CLIP-I — Resilience (recovered vs baseline target) ===")
clip_i_resilience = clip_i_score(Path("outputs/recovered"), Path("outputs/baseline/target"))
print(f"  CLIP-I resilience: {clip_i_resilience:.4f}")
print(f"  (Lower = recovered images further from baseline Van Gogh = recovery failed = robust erasure)")

=== CLIP-I — Efficacy (erased target vs baseline target) ===
  CLIP-I efficacy: 0.5652
  (Lower = more change from baseline = good erasure)

=== CLIP-I — Resilience (recovered vs baseline target) ===
  CLIP-I resilience: 0.3840
  (Lower = recovered images further from baseline Van Gogh = recovery failed = robust erasure)
